# Chapter 2.2 - Tokenizing text

In [1]:
with open("the-verdict.txt","r",encoding="utf-8") as f:
    raw_text=f.read()

In [2]:
raw_text[:500]

'I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)\n\n"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it\''

In [3]:
len(raw_text)

20479

##### using a simple example text to split a text on whitespace characters using regular expression

In [4]:
import re
text= "hello world. This is a test."
result= re.split(r'(\s)', text)
print(result)

['hello', ' ', 'world.', ' ', 'This', ' ', 'is', ' ', 'a', ' ', 'test.']


##### some words are still connected to punctuation characters that we want to have as separate list entries.

##### let's modify the regular expression splits on whitespaces (\s) and commas, and periods

In [5]:
result= re.split(r'([,.]|\s)',text)
print(result)

['hello', ' ', 'world', '.', '', ' ', 'This', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


##### a small remaining issue is that the list still includes whitespace characters. Optionally, we can remove these redundant characters 

result=[item for item in result  if item.strip()]
print(result)

##### let's modify it a bit further so that it can also handle other types of punctuation, such asquestion marks, quotation marks, and the double-dashes 

In [6]:
text="Hello, world. Is this-- a test?"
result=re.split(r'([,.:;?_!"()\']|--|\s)',text)
result=[item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


##### Now that we got a basic tokenizer working, let's apply it to Edith Wharton's entire short  story

In [7]:
preprocessed= re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed=[item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed))

4690


In [8]:
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


# Chapter 2.3 - Converting tokens into token IDs

##### Let's now create a list of all unique tokens and sort them alphabetically to determine the vocabulary size

In [9]:
all_words= sorted(set(preprocessed)) #set removes duplicates, sorted sorts the unique tokens into alphabetical order

In [10]:
all_words[:20] 

['!',
 '"',
 "'",
 '(',
 ')',
 ',',
 '--',
 '.',
 ':',
 ';',
 '?',
 'A',
 'Ah',
 'Among',
 'And',
 'Are',
 'Arrt',
 'As',
 'At',
 'Be']

In [11]:
vocab_size=len(all_words)
print(vocab_size)

1130


In [12]:
vocab={token:integer for integer, token in enumerate(all_words)}

In [13]:
#print the first 25 vocab
for i, item in enumerate(vocab.items()):
    print(item)
    if i>25:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)


##### Implementing a simple text tokenizer

In [14]:
class SimpleTokenizerV1:
    def __init__(self,vocab):
        self.str_to_int= vocab #token to id  mapping-> for encoding
        self.int_to_str= {i:s for s,i in vocab.items()} # we swapped it here, so id to token mapping for decoding
        
    def encode(self,text):
        preprocessed=re.split(r'([,.?_!"()\']|--|\s)',text) #splits the texts 
        preprocessed=[item.strip() for item in preprocessed if item.strip()] #removes any whitespace characters.
        ids=[self.str_to_int[s] for s in preprocessed] #look up to each word in preprocessed and find its coresponding ids
        return ids # return id?

    def decode(self,ids):
        text= " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

##### instantiate the class

In [15]:
tokenizer=SimpleTokenizerV1(vocab)

##### lets use a sample text to use our class encode method to encode a text into ids

In [16]:
text=""""It's the last he painted, you know," Mrs. Gisburn said with pardonable pride."""
ids=tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


##### decode back to the original text.

In [17]:
print(tokenizer.decode(ids))

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


##### So far, so good. We implemented a tokenizer capable of tokenizing and de-tokenizingtext based on a snippet from the training set. Let's now apply it to a new text sample that is not contained in the training set:

In [18]:
#text="hello, do you like tea?"
#print(tokenizer.encode(text))

##### The problem is that the word "Hello" was not used in the The Verdict short story. Hence, it is not contained in the vocabulary. This highlights the need to consider large and diverse training sets to extend the vocabulary when working on LLMs.

##### In the next section, we will test the tokenizer further on text that contains unknown words, and we will also discuss additional special tokens that can be used to provide further context for an LLM during training.

# Chapter 2.4 - Adding special context tokens

##### In this section, we will modify this tokenizer to handle unknown words.

##### we will modify the vocabulary and tokenizer we implemented in the previous section, SimpleTokenizerV2, to support two new tokens, <|unk|> and <|endoftext|>

##### |unk| stands for unknown, used when the tokenizer encounters a word that isn't in the vocabulary.

##### it replaces the unknown word in the vocabulary with itself

##### |endoftext| marks where one piece of text ends. A special token that marks the end of one document/text and the beginning of the next, helping the model recognize text boundaries during training and generation.

In [19]:
all_tokens= sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>","<|unk|>"])
vocab={token:integer for integer,token in enumerate(all_tokens)}

In [20]:
for i, item in enumerate(list(vocab.items())[-5:]): #print the last 5 elements
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [21]:
vocab_size=len(vocab)
print(vocab_size)

1132


##### update the tokenizer class to v2 using the new added special tokens

In [22]:
class SimpleTokenizerV2:
    def __init__(self,vocab):
        self.str_to_int= vocab
        self.int_to_str={i:s for s,i in vocab.items()}

    def encode(self,text):
        preprocessed=re.split(r'([,.?_!"()\']|--|\s)', text)
        preprocessed=[item.strip() for item in preprocessed if item.strip()]
        #replace unknown word in vocab with unk.
        preprocessed=[item if item in self.str_to_int else "<|unk|>" for item in preprocessed]
        ids=[self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self,ids):
        text= " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

##### one thing to notice here is that we didn't use <|endoftext|> cause <|endoftext|> is usually added during dataset preparation, before tokenization,when preparing the data

#### creating a sample dataset for testing the v2 tokenizer.

In [23]:
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text=" <|endoftext|> ".join((text1,text2))
print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [24]:
tokenizer=SimpleTokenizerV2(vocab)

In [25]:
print(tokenizer.encode(text))

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]


In [26]:
tokenized_ids=tokenizer.encode(text)

In [27]:
print(tokenizer.decode(tokenized_ids))

<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


##### as we can see, the tokenizer replaced the unknown word in vocab, fixing the issue we had in v1.

##### More special tokens used in LLMs: **[BOS] (Beginning of Sequence)** marks the start of a text, **[EOS] (End of Sequence)** marks the end of a text or separates multiple documents (similar to `<|endoftext|>`), and **[PAD] (Padding)** extends shorter sequences in a batch so all have equal length. In GPT, `<|endoftext|>` is mainly used as the end-of-text/document marker and can also serve as a padding token because attention masks ignore padded positions. GPT does **not** use an `<|unk|>` token; instead, it uses **Byte Pair Encoding (BPE)** to split unseen words into known subword tokens.

# Chapter 2.5 - Byte pair encoding

In [28]:
%pip install tiktoken

Note: you may need to restart the kernel to use updated packages.


In [30]:
from importlib.metadata import version
import tiktoken
print("tiktoken version:", version("tiktoken"))

tiktoken version: 0.13.0


##### instantiate the bpe tokenizer

In [31]:
tokenizer=tiktoken.get_encoding("gpt2")

In [32]:
text = "Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace."
integers= tokenizer.encode(text,allowed_special={"<|endoftext|>"})
print(integers)         

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]


In [33]:
strings=tokenizer.decode(integers)
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.


# Chapter 2.6 - Data Sampling with a Sliding Window

In [2]:
with open("the-verdict.txt","r",encoding="utf-8") as f:
    raw_text= f.read()

In [3]:
import tiktoken
tokenizer=tiktoken.get_encoding("gpt2")
enc_text= tokenizer.encode(raw_text)
print(len(enc_text))

5145


##### we remove the first 50 tokens from the dataset for demonstration purposes as it results in a slightly more interesting text passage in the next steps:

In [4]:
enc_sample=enc_text[50:]

In [5]:
print(enc_sample)

[290, 4920, 2241, 287, 257, 4489, 64, 319, 262, 34686, 41976, 13, 357, 10915, 314, 2138, 1807, 340, 561, 423, 587, 10598, 393, 28537, 2014, 198, 198, 1, 464, 6001, 286, 465, 13476, 1, 438, 5562, 373, 644, 262, 1466, 1444, 340, 13, 314, 460, 3285, 9074, 13, 46606, 536, 5469, 438, 14363, 938, 4842, 1650, 353, 438, 2934, 489, 3255, 465, 48422, 540, 450, 67, 3299, 13, 366, 5189, 1781, 340, 338, 1016, 284, 3758, 262, 1988, 286, 616, 4286, 705, 1014, 510, 26, 475, 314, 836, 470, 892, 286, 326, 11, 1770, 13, 8759, 2763, 438, 1169, 2994, 284, 943, 17034, 318, 477, 314, 892, 286, 526, 383, 1573, 11, 319, 9074, 13, 536, 5469, 338, 11914, 11, 33096, 663, 4808, 3808, 62, 355, 996, 484, 547, 12548, 287, 281, 13079, 410, 12523, 286, 22353, 13, 843, 340, 373, 407, 691, 262, 9074, 13, 536, 48819, 508, 25722, 276, 13, 11161, 407, 262, 40123, 18113, 544, 9325, 701, 11, 379, 262, 938, 402, 1617, 261, 12917, 905, 11, 5025, 502, 878, 402, 271, 10899, 338, 366, 31640, 12, 67, 20811, 1, 284, 910, 11, 351, 10

##### One of the easiest and most intuitive ways to create the input-target pairs for the nextword prediction task is to create two variables, x and y, where x contains the input tokens and y contains the targets, which are the inputs shifted by 1:

In [ ]:
[290(0), 4920(1), 2241(2), 287(3), 257(4), 4489(5)]

In [11]:
context_size= 4
# the context size determines how many tokens are includede in the input
x=enc_sample[:context_size] 
y=enc_sample[1:context_size + 1] right sift by 1.
print(f"x: {x}")
print(f"y: {y}")

x: [290, 4920, 2241, 287]
y: [4920, 2241, 287, 257]


### Explanation of the above code and the reason why we shift by 1.
```python
context_size = 4

# The context size determines how many tokens are included in the input sequence.
x = enc_sample[:context_size]
y = enc_sample[1:context_size + 1]

print(f"x: {x}")
print(f"y: {y}")
```

### What is happening?

Language models such as GPT are trained using the **next-token prediction** task. Given a sequence of input tokens, the model learns to predict the token that comes immediately after them.

Suppose our tokenized sequence is:

```python
enc_sample = [290, 4920, 2241, 287, 257, 4489]
```

| Index | 0 | 1 | 2 | 3 | 4 | 5 |
|------:|--:|--:|--:|--:|--:|--:|
| Token |290|4920|2241|287|257|4489|

---

### Creating the input sequence (`x`)

```python
x = enc_sample[:context_size]
```

Since `context_size = 4`, this becomes:

```python
x = enc_sample[:4]
```

Python slices **include the start index and exclude the stop index**, so this selects the tokens at indices:

```
0, 1, 2, 3
```

Result:

```python
x = [290, 4920, 2241, 287]
```

---

### Creating the target sequence (`y`)

```python
y = enc_sample[1:context_size + 1]
```

Since `context_size = 4`, this becomes:

```python
y = enc_sample[1:5]
```

This selects the tokens at indices:

```
1, 2, 3, 4
```

Result:

```python
y = [4920, 2241, 287, 257]
```

---

### Why do we shift by one token?

The target sequence is simply the input sequence **shifted one position to the right**.

```
Input (x)

290    4920    2241    287
 │       │       │       │
 ▼       ▼       ▼       ▼
4920    2241     287    257

Target (y)
```

Each token in `y` is the **next token** after the corresponding token in `x`.

For example:

| Input Token | Target Token |
|-------------|--------------|
| 290 | 4920 |
| 4920 | 2241 |
| 2241 | 287 |
| 287 | 257 |

This teaches the model:

- After `290`, predict `4920`.
- After `4920`, predict `2241`.
- After `2241`, predict `287`.
- After `287`, predict `257`.

---

### Why use `context_size + 1`?

Python slices **exclude the stop index**.

If we wrote:

```python
y = enc_sample[1:context_size]
```

it would become:

```python
y = enc_sample[1:4]
```

which returns:

```python
[4920, 2241, 287]
```

This has only **3 tokens**, while `x` has **4 tokens**.

To keep `x` and `y` the same length, we extend the stop index by one:

```python
y = enc_sample[1:context_size + 1]
```

This produces **4 target tokens**, giving each input token a corresponding "next token" target.

---

### Summary

- `x` contains the current context (input tokens).
- `y` contains the same sequence shifted one token to the right.
- Every token in `y` is the correct next token for the corresponding token in `x`.
- This is how GPT and other autoregressive language models learn to predict the next token in a sequence.
````


In [13]:
for i in range(1,context_size+1):
    context=enc_sample[0:i]
    desired=enc_sample[i]
    print(context,"--------->",desired)

[290] ---------> 4920
[290, 4920] ---------> 2241
[290, 4920, 2241] ---------> 287
[290, 4920, 2241, 287] ---------> 257


##### Everything left of the arrow (---->) refers to the input an LLM would receive, and the tokenID on the right side of the arrow represents the target token ID that the LLM is supposed to predict.


##### let's repeat the previous code but convert the token IDs into text:

In [21]:
for  i in range(1,context_size+1):
    context=enc_sample[0:i]
    desired=enc_sample[i]
    print(tokenizer.decode(context),"--------->",tokenizer.decode([desired]))

 and --------->  established
 and established --------->  himself
 and established himself --------->  in
 and established himself in --------->  a


##### We wrap desired in [] because tokenizer.decode() expects a list of token IDs, even when decoding a single token.